In [40]:
import os 
os.environ["OMP_NUM_THREADS"] = "50"

In [41]:
import numpy as np
import healpy as hp
import astropy.units as u
from pathlib import Path
from ds_utils import  *
from ps_utils import  *

In [42]:
DATA_DIR = Path("/pscratch/sd/k/kp22/deft/")

In [43]:
RES = 256
STEP_SIZE = 6 * u.deg
ANG_X = 6. * u.deg
ANG_Y = 6. * u.deg
ptsrc = 2
flatskymapparams = [RES, RES, 60.*ANG_X.value/RES, 60.*ANG_Y.value/RES] #Code requires pixelres to be in arcmin
flatskymapparams

[256, 256, np.float64(1.40625), np.float64(1.40625)]

# Creating training data  (Skip this section if the maps are already saved)

## 1. Loading the point source masked maps
These maps are same as the AGORA maps but the point sources are masked at a threshold of 2mJ/6mJy and those points are replaced by zeroes.

These files are available at: /sptlocal/analysis/ymap/sims/mdpl2/data/v0.7/bahamas80_scal1.000/mask_radio_cib_6.0mjy/cib

## 2. Preprocessing the maps
Before we create training patches, we need to low pass filter these maps so that we don't have higher frequency modes aliasing into lower frequency modes. To do this we apply a sharp cutoff for modes above 7000. 
I noticed that there are a few pixels that have negative values that doesn't make physical sense. I believe they are a result of some filtering process. We can safely zero them out as they are only a fraction of a percent.

## 3. Creating map patches
The patches are created by laying down a set of centroids $(l_i + s, b_i +\frac{s}{\cos(l_i)})$ (longitude,latitude), where $s$ is a step size parameter, and $\frac{s}{\cos(l_i)}$ is a step between longitudes for a given latitude, which ensures the same angular separation in the latitudinal direction. Each centroid is then rotated to the equator, and an $6 \times 6$ square degrees region around the centroid is projected onto a cartesian grid with 256 pixels along each size.

### a. Loading the point source masked maps

In [48]:
cib_150_map = hp.read_map(Path(DATA_DIR/ f"mask_radio_cib_{ptsrc}mJy" / "mdpl2_150GHz_fullsky.fits"))
#cib_150_map = hp.read_map(Path(DATA_DIR/ f"mask_radio_cib_{ptsrc}mJy" / "tsz" / "mdpl2_150GHz_fullsky_mask_3e+14.fits"))

### b. Low pass filtering

In [52]:
fullsky_map = np.copy(cib_150_map)

In [53]:
fullsky_alm = hp.map2alm(fullsky_map,lmax=13000)
ell_cut = 7000
lmax=13000
alm_filtered = hp.sphtfunc.almxfl(fullsky_alm, [1 if ell <= ell_cut else 0 for ell in range(lmax + 1)])

In [54]:
fullsky_map = hp.sphtfunc.alm2map(alm_filtered, nside=8192)

In [56]:
fullsky_map[fullsky_map<0]=0 # For CIB
#fullsky_map[fullsky_map>0]=0 # For tSZ

In [57]:
hp.write_map(Path(DATA_DIR/ f"mask_radio_cib_{ptsrc}mJy" / "mdpl2_150GHz_fullsky_lmax7000.fits"), fullsky_map)

setting the output map dtype to [dtype('float64')]


### c. Creating map patches

In [ ]:
# get centroids
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))

In [ ]:
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, fullsky_map) for (lon, lat) in centers], dtype=np.float32)

In [20]:
fname = f"data/low_pass/{ptsrc}mJy/cut_maps_RES_{RES}_ANG_X_{ANG_X.value}_deg_{ptsrc}mJy_lp.npy"
np.save(fname,cut_maps)

## 4. MinMax Normalization

Let each map be denoted by $m(x, y)$. 
We normalize (MinMax) the spatial domain maps to have values between 0 and 1. Here $\vec{m}$ denotes the collection of maps.
$$m = \frac{m - \min(\vec{m})}{\max(\vec{m}) - \min(\vec{m})}$$

In [33]:
cut_maps= np.load(fname)

In [34]:
def apply_maxmin_normalization(maps):
    min_val = np.min(maps)
    max_val = np.max(maps)
    return (maps - min_val) / (max_val - min_val) 

In [35]:
processed_maps = np.copy(cut_maps)

In [36]:
min_val = np.min(cut_maps)
max_val = np.max(cut_maps)
print (min_val,max_val - min_val)  

0.0 110.732285


In [37]:
processed_maps = apply_maxmin_normalization(processed_maps)

In [38]:
fpath = "data/low_pass/{:d}mJy/CIB_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_zero_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
#fpath = "data/low_pass/{:d}mJy/tSZ3_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_norm_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
print(fpath)

data/low_pass/2mJy/CIB_map_150GHz_256_st6_minmax_2mJy_zero_lp.npy


In [39]:
np.save(fpath, processed_maps)

# Gaussian patches

In [ ]:
cib_map_meansub = cib_150_map - np.mean(cib_150_map)
cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)

In [ ]:
gaussian_map = hp.synfast(cl_from_map, 8192)

In [ ]:
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))

In [ ]:
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, gaussian_map) for (lon, lat) in centers], dtype=np.float32)

In [ ]:
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz.npy",cut_maps)

# Joint gaussian maps

In [ ]:
cib_map_meansub = cib_150_map - np.mean(cib_150_map)
cib_cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)

In [ ]:
tsz_map_meansub = tsz_150_map - np.mean(tsz_150_map)
tsz_cl_from_map = hp.anafast(tsz_map_meansub, lmax=13000)

In [ ]:
cib_tsz_cl_from_map = hp.anafast(cib_map_meansub,tsz_map_meansub, lmax=13000)

In [ ]:
hp.write_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits",tsz_cl_from_map)

In [ ]:
hp.write_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits",cib_cl_from_map)

In [ ]:
hp.write_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits",cib_tsz_cl_from_map)

In [ ]:
cib_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits")

In [ ]:
tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits")

In [ ]:
cib_tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits")

In [ ]:
# Ordering for healpy.synfast:
# [C_ℓ^TT (aa), C_ℓ^EE (bb), C_ℓ^BB (ignored), C_ℓ^TE (ab)]
cls = [cib_cl_from_map, tsz_cl_from_map, np.zeros_like(cib_cl_from_map), cib_tsz_cl_from_map]
maps = hp.synfast(cls, nside=8192, new=True, pol=False)
cib_map = maps[0]
tsz_map = maps[1]

In [ ]:
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, cib_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_cib_joint3.npy",cut_maps)

In [ ]:
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, tsz_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz_joint3.npy",cut_maps)